In [1]:
# imports
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [2]:
# loading the model
class MLPExperiment1(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 2)
        )

    def forward(self, x):

        return self.network(x)

In [3]:
model = MLPExperiment1()

model.load_state_dict(
    torch.load("mlp_exp1_model.pth", map_location=torch.device("cpu"))
)

model.eval()

print("MLP Loaded")

MLP Loaded


In [4]:
# loading the scaler
scaler = joblib.load("mlp_scaler_exp1.pkl")

print("Scaler Loaded")

Scaler Loaded


In [5]:
# loading datasets
datasets = {
    "Degraded": "degraded_mlp.csv",
    "Clean": "clean_mlp.csv",
    "Natural": "natural_mlp.csv",
    "Unknown": "unknown_mlp.csv",
}

In [6]:
# evaluation function
def evaluate_dataset(csv_file):

    df = pd.read_csv(csv_file)

    X = df[["quality_score", "best_similarity", "margin"]]

    y_true = df["label"]

    X_scaled = scaler.transform(X)

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    with torch.no_grad():

        outputs = model(X_tensor)

        y_pred = torch.argmax(outputs, dim=1).numpy()

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(y_true, y_pred, zero_division=0)

    recall = recall_score(y_true, y_pred, zero_division=0)

    f1 = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)

    if cm.shape == (2, 2):

        TN, FP, FN, TP = cm.ravel()

        TAR = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        FRR = FN / (TP + FN) if (TP + FN) > 0 else np.nan

        TRR = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        FAR = FP / (TN + FP) if (TN + FP) > 0 else np.nan

    else:

        TAR = np.nan
        FRR = np.nan
        TRR = np.nan
        FAR = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TAR": TAR,
        "FRR": FRR,
        "TRR": TRR,
        "FAR": FAR,
    }

In [7]:
# threshold evaluation
def evaluate_threshold(csv_file):

    df = pd.read_csv(csv_file)

    y_true = df["label"]

    threshold_pred = (df["best_similarity"] >= 0.6).astype(int)

    accuracy = accuracy_score(y_true, threshold_pred)

    precision = precision_score(y_true, threshold_pred, zero_division=0)

    recall = recall_score(y_true, threshold_pred, zero_division=0)

    f1 = f1_score(y_true, threshold_pred, zero_division=0)

    cm = confusion_matrix(y_true, threshold_pred)

    if cm.shape == (2, 2):

        TN, FP, FN, TP = cm.ravel()

        TAR = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        FRR = FN / (TP + FN) if (TP + FN) > 0 else np.nan

        TRR = TN / (TN + FP) if (TN + FP) > 0 else np.nan

        FAR = FP / (TN + FP) if (TN + FP) > 0 else np.nan

    else:

        TAR = np.nan
        FRR = np.nan
        TRR = np.nan
        FAR = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TAR": TAR,
        "FRR": FRR,
        "TRR": TRR,
        "FAR": FAR,
    }

In [8]:
# running our multi layer MLP
mlp_multi_results = {}

for dataset_name, csv_file in datasets.items():

    result = evaluate_dataset(csv_file)

    mlp_multi_results[dataset_name] = result

    print("\n======================")
    print(dataset_name)
    print("======================")

    print(pd.DataFrame([result]).round(4))


Degraded
   Accuracy  Precision  Recall      F1     TAR     FRR     TRR     FAR
0    0.9704     0.9981  0.9716  0.9846  0.9716  0.0284  0.9167  0.0833

Clean
   Accuracy  Precision  Recall      F1     TAR     FRR  TRR  FAR
0    0.9704        1.0  0.9697  0.9846  0.9697  0.0303  1.0  0.0

Natural
   Accuracy  Precision  Recall    F1  TAR  FRR  TRR  FAR
0     0.619        1.0     0.6  0.75  0.6  0.4  1.0  0.0

Unknown
   Accuracy  Precision  Recall   F1  TAR  FRR  TRR  FAR
0       1.0        0.0     0.0  0.0  NaN  NaN  NaN  NaN


/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [9]:
# running fixed threshold method
threshold_results = {}

for dataset_name, csv_file in datasets.items():

    result = evaluate_threshold(csv_file)

    threshold_results[dataset_name] = result

    print("\n======================")
    print(dataset_name)
    print("======================")

    print(pd.DataFrame([result]).round(4))


Degraded
   Accuracy  Precision  Recall      F1     TAR     FRR  TRR  FAR
0    0.6037        1.0  0.5947  0.7458  0.5947  0.4053  1.0  0.0

Clean
   Accuracy  Precision  Recall      F1     TAR     FRR  TRR  FAR
0    0.7333        1.0  0.7273  0.8421  0.7273  0.2727  1.0  0.0

Natural
   Accuracy  Precision  Recall      F1    TAR    FRR  TRR  FAR
0     0.119        1.0   0.075  0.1395  0.075  0.925  1.0  0.0

Unknown
   Accuracy  Precision  Recall   F1  TAR  FRR  TRR  FAR
0       1.0        0.0     0.0  0.0  NaN  NaN  NaN  NaN


/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [13]:
# svm and single layer mlp metrics (already tested earlier)
mlp_single_results = {
    "Degraded": {
        "Accuracy": 0.938889,
        "Precision": 1.0,
        "Recall": 0.9375,
        "F1": 0.967742,
        "TAR": 0.9375,
        "FRR": 0.0625,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Clean": {
        "Accuracy": 0.9407,
        "Precision": 1.0,
        "Recall": 0.9394,
        "F1": 0.9688,
        "TAR": 0.9394,
        "FRR": 0.0606,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Natural": {
        "Accuracy": 0.3810,
        "Precision": 1.0,
        "Recall": 0.3500,
        "F1": 0.5185,
        "TAR": 0.3500,
        "FRR": 0.6500,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Unknown": {"TRR": 1.0, "FAR": 0.0, "Correct Rejects": 104, "False Accepts": 0},
}

svm_results = {
    "Degraded": {
        "Accuracy": 0.974074,
        "Precision": 0.998062,
        "Recall": 0.975379,
        "F1": 0.986590,
        "TAR": 0.975379,
        "FRR": 0.024621,
        "TRR": 0.916667,
        "FAR": 0.083333,
    },
    "Clean": {
        "Accuracy": 0.970370,
        "Precision": 0.99,
        "Recall": 0.9773,
        "F1": 0.9800,
        "TAR": 0.9773,
        "FRR": 0.0227,
        "TRR": 0.6667,
        "FAR": 0.3333,
    },
    "Natural": {
        "Accuracy": 0.554422,
        "Precision": 1.0,
        "Recall": 0.532143,
        "F1": 0.694639,
        "TAR": 0.532143,
        "FRR": 0.467857,
        "TRR": 1.0,
        "FAR": 0.0,
    },
    "Unknown": {"TRR": 1.0, "FAR": 0.0, "Correct Rejects": 104, "False Accepts": 0},
}

In [15]:
# final master comparison table
master_rows = []

for dataset_name in datasets.keys():

    for method_name, result_dict in [
        ("Threshold", threshold_results[dataset_name]),
        ("MLP Single Layer", mlp_single_results[dataset_name]),
        ("MLP Multi Layer", mlp_multi_results[dataset_name]),
        ("SVM", svm_results[dataset_name]),
    ]:

        row = {"Dataset": dataset_name, "Method": method_name}

        row.update(result_dict)

        master_rows.append(row)

master_table = pd.DataFrame(master_rows)

master_table.round(4)

,Dataset,Method,Accuracy,Precision,Recall,F1,TAR,FRR,TRR,FAR,Correct Rejects,False Accepts
0,Degraded,Threshold,0.6037,1.0000,0.5947,0.7458,0.5947,0.4053,1.0000,0.0000,NaN,NaN
1,Degraded,MLP Single Layer,0.9389,1.0000,0.9375,0.9677,0.9375,0.0625,1.0000,0.0000,NaN,NaN
2,Degraded,MLP Multi Layer,0.9704,0.9981,0.9716,0.9846,0.9716,0.0284,0.9167,0.0833,NaN,NaN
3,Degraded,SVM,0.9741,0.9981,0.9754,0.9866,0.9754,0.0246,0.9167,0.0833,NaN,NaN
4,Clean,Threshold,0.7333,1.0000,0.7273,0.8421,0.7273,0.2727,1.0000,0.0000,NaN,NaN
5,Clean,MLP Single Layer,0.9407,1.0000,0.9394,0.9688,0.9394,0.0606,1.0000,0.0000,NaN,NaN
6,Clean,MLP Multi Layer,0.9704,1.0000,0.9697,0.9846,0.9697,0.0303,1.0000,0.0000,NaN,NaN
7,Clean,SVM,0.9704,0.9900,0.9773,0.9800,0.9773,0.0227,0.6667,0.3333,NaN,NaN
8,Natural,Threshold,0.1190,1.0000,0.0750,0.1395,0.0750,0.9250,1.0000,0.0000,NaN,NaN
9,Natural,MLP Single Layer,0.3810,1.0000,0.3500,0.5185,0.3500,0.6500,1.0000,0.0000,NaN,NaN
